In [1]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from sklearn.utils import resample
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')
mean_spf_trim_nr = pd.read_csv('cleaned_data/mean_spf_trim_nr.csv')
ind_spf_trim_nr = pd.read_csv('cleaned_data/ind_spf_trim_nr.csv')

mean_spf_trim['DATE'] = pd.to_datetime(mean_spf_trim['DATE'])
mean_spf_trim_nr['DATE'] = pd.to_datetime(mean_spf_trim_nr['DATE'])
ind_spf_trim['DATE'] = pd.to_datetime(ind_spf_trim['DATE'])
ind_spf_trim_nr['DATE'] = pd.to_datetime(ind_spf_trim_nr['DATE'])

ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')
ind_spf_trim_nr = ind_spf_trim_nr.dropna(subset='r_t1')

###############################################
                ## ESTIMATION ##
###############################################

### OLS ###
def OLS(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        revisions = df[f'r_t{j}']
        revisions = sm.add_constant(revisions)
        errors = df[f'e_t{j}']
        initial = sm.OLS(errors, revisions).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=None))
    return regs


### ID FE ###
def ID_FE(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=None))
    return regs


### Two-way Fixed Effects ###
def T_FE(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['DATE'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        regs.append(initial.get_robustcov_results(cov_type='HC2'))
    return regs


### AR Estimates ###
def AR(df, end_date):
    df = df.loc[(df['DATE'] >= '1965-06-30') & (df['DATE'] <= end_date)]  # Filter data
    grwth_t = df['t3']
    reg = AutoReg(grwth_t, 1).fit()
    return reg


### Parameter Estimates ###
def params(ols, pldols, fe, fe2, ar1):
    params = []
    params.append((1+ols) / (1+(1+ols)))
    params.append(1 / (1 + (1+ols)))
    params.append((-((2 * pldols) + 1) + np.sqrt(((2 * pldols) + 1)**2 - 4 * (pldols + (pldols * ar1**2) + 1) * pldols)) / (2 * (pldols + (pldols * ar1**2) + 1)))
    params.append((-((2 * fe) + 1) + np.sqrt(((2 * fe) + 1)**2 - 4 * (fe + (fe * ar1**2) + 1) * fe)) / (2 * (fe + (fe * ar1**2) + 1)))
    params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))
    return params


def compute_regs(date, mean, ind):
    mean_regs = OLS(mean, f'{date}')
    ind_regs = OLS(ind, f'{date}')
    ind_regs_fe = ID_FE(ind, f'{date}')
    ind_regs_fe2 = T_FE(ind, f'{date}')
    ar_1 = AR(vintage_trim, f'{date}')
    regs = [mean_regs, ind_regs, ind_regs_fe, ind_regs_fe2, ar_1]
    parameters = params(mean_regs[3].params[1], ind_regs[3].params[1], ind_regs_fe[3].params[1], ind_regs_fe2[3].params[1], ar_1.roots)
    return regs, parameters

regs_2020, parameters_2020 = compute_regs('2020-06-30', mean_spf_trim, ind_spf_trim)
regs_2020_nr, parameters_2020_nr = compute_regs('2020-06-30', mean_spf_trim_nr, ind_spf_trim_nr)
%store regs_2020
%store parameters_2020
%store regs_2020_nr
%store parameters_2020_nr

regs_2022, parameters_2022 = compute_regs('2022-12-31', mean_spf_trim, ind_spf_trim)
regs_2022_nr, parameters_2022_nr = compute_regs('2022-12-31', mean_spf_trim_nr, ind_spf_trim_nr)
%store regs_2022
%store parameters_2022
%store regs_2022_nr
%store parameters_2022_nr

/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/var/folders/6q/zww1c4xj04d9c7jlxyz1zc6c0000gn/T/ipykernel_62779/2358732832.py:84: RuntimeWarning: invalid value encountered in sqrt
  params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))
/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Stored 'regs_2020' (list)
Stored 'parameters_2020' (list)
Stored 'regs_2020_nr' (list)
Stored 'parameters_2020_nr' (list)


/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/var/folders/6q/zww1c4xj04d9c7jlxyz1zc6c0000gn/T/ipykernel_62779/2358732832.py:84: RuntimeWarning: invalid value encountered in sqrt
  params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))
/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Stored 'regs_2022' (list)
Stored 'parameters_2022' (list)
Stored 'regs_2022_nr' (list)
Stored 'parameters_2022_nr' (list)


In [4]:
regs_2020[1][3].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t3   R-squared:                       0.018
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     13.13
Date:                Mon, 29 Apr 2024   Prob (F-statistic):           0.000295
Time:                        12:29:53   Log-Likelihood:                 11418.
No. Observations:                3775   AIC:                        -2.283e+04
Df Residuals:                    3773   BIC:                        -2.282e+04
Df Model:                           1                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0024      0.001     -4.651      0.000      -0.003      -0.001
r_t3          -0.2420      0.067     -3.623      0.000      -0.373      -0.111
==============================================================================
Omnibus:                      146.212   Durbin-Watson:                   0.420
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              339.589
Skew:                          -0.219   Prob(JB):                     1.82e-74
Kurtosis:                       4.403   Cond. No.                         150.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using None lags and without small sample correction
"""

In [8]:
###############################################
              ## ID BOOTSTRAP ##
###############################################
def boot(df, R, date):
    RR = R + 1
    boot_regs0 = []
    boot_regs1 = []
    boot_regs2 = []
    boot_regs3 = []
    for i in range(1, RR):
        boot = resample(df['ID'], replace=True, n_samples=len(df), random_state=i)
        boot_df = df[df['ID'].isin(boot)]
        boot_ols = OLS(boot_df, date)
        boot_regs0.append(boot_ols[0].params[1])
        boot_regs1.append(boot_ols[1].params[1])
        boot_regs2.append(boot_ols[2].params[1])
        boot_regs3.append(boot_ols[3].params[1])
    boot_regs = [boot_regs0, boot_regs1, boot_regs2, boot_regs3]
    return boot_regs

boot_regs_2020 = boot(ind_spf_trim, 10000, '2020-06-30')
%store boot_regs_2020

boot_regs_2022 = boot(ind_spf_trim, 10000, '2022-12-31')
%store boot_regs_2022

Stored 'boot_regs_2020' (list)
Stored 'boot_regs_2022' (list)


In [9]:
%store -r boot_regs_2020
%store -r boot_regs_2022
print(np.quantile(boot_regs_2020[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2020[3], 0.5))
print(np.quantile(boot_regs_2022[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2022[3], 0.5))

[-0.24410015 -0.24175651]
-0.2420227372829345
[0.0602634 0.0625437]
0.062246529362919856
